In [ ]:
import os
import unittest
from transformers import AutoModelForCausalLM, AutoTokenizer
from llmcompressor.modifiers.quantization import QuantizationModifier
from llmcompressor.transformers import oneshot

class TestQuantizationProcess(unittest.TestCase):

    def test_quantization_and_save(self):
        # Load model
        model_stub = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
        model_name = model_stub.split("/")[-1]

        model = AutoModelForCausalLM.from_pretrained(
            model_stub,
            torch_dtype="auto",
        )

        tokenizer = AutoTokenizer.from_pretrained(model_stub)

        # Configure the quantization algorithm and scheme
        recipe = QuantizationModifier(
            targets="Linear",
            scheme="FP8_DYNAMIC",
            ignore=["lm_head"],
        )

        # Apply quantization
        oneshot(
            model=model,
            recipe=recipe,
        )

        # Save to disk in compressed-tensors format
        save_path = model_name + "-FP8-dynamic"
        model.save_pretrained(save_path)
        tokenizer.save_pretrained(save_path)

        # Assertions to verify save
        self.assertTrue(os.path.exists(save_path), f"Save path does not exist: {save_path}")
        self.assertTrue(os.path.exists(os.path.join(save_path, "config.json")), "Model config not found")
        self.assertTrue(os.path.exists(os.path.join(save_path, "tokenizer_config.json")), "Tokenizer config not found")

unittest.main(argv=[''], verbosity=2, exit=False)


In [ ]:
# Pillow (PIL) smoke test — validates CVE override (RHAIENG-3209) and image workflows
# Used for: image preprocessing, visualization, dataset handling in ML pipelines
import unittest
from PIL import Image
import torch
from torchvision import transforms

class TestPillowImageWorkflow(unittest.TestCase):

    def test_pillow_version_and_basic_ops(self):
        from PIL import __version__ as pil_version
        self.assertGreaterEqual(
            tuple(int(x) for x in pil_version.split(".")[:3]),
            (12, 1, 1),
            f"Pillow >= 12.1.1 required (CVE-2026-25990); got {pil_version}",
        )

    def test_image_create_resize_roundtrip(self):
        img = Image.new("RGB", (100, 100), color="red")
        img_small = img.resize((50, 50), Image.Resampling.LANCZOS)
        self.assertEqual(img_small.size, (50, 50))

    def test_torchvision_transform_with_pillow(self):
        img = Image.new("RGB", (64, 64), color="blue")
        to_tensor = transforms.ToTensor()
        t = to_tensor(img)
        self.assertEqual(t.shape, (3, 64, 64))

unittest.main(argv=[""], verbosity=2, exit=False)